In [1]:
import os

from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression, SGDRegressor, Ridge, LogisticRegression, Lasso
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, classification_report, roc_auc_score
import joblib
import pandas as pd
import numpy as np

In [2]:
"""
线性回归直接预测房子价格
:return: None
"""
# 获取数据
fe_cal = fetch_california_housing(data_home='data')

print("获取特征值")
print(fe_cal.data.shape)
print('-' * 50)
print(fe_cal.data[0])
print("目标值")
print(fe_cal.target) #单位是10万美金
print(fe_cal.DESCR)
print('-' * 50)
print(fe_cal.feature_names) #特征列的名字

获取特征值
(20640, 8)
--------------------------------------------------
[   8.3252       41.            6.98412698    1.02380952  322.
    2.55555556   37.88       -122.23      ]
目标值
[4.526 3.585 3.521 ... 0.923 0.847 0.894]
.. _california_housing_dataset:

California Housing dataset
--------------------------

**Data Set Characteristics:**

:Number of Instances: 20640

:Number of Attributes: 8 numeric, predictive attributes and the target

:Attribute Information:
    - MedInc        median income in block group
    - HouseAge      median house age in block group
    - AveRooms      average number of rooms per household
    - AveBedrms     average number of bedrooms per household
    - Population    block group population
    - AveOccup      average number of household members
    - Latitude      block group latitude
    - Longitude     block group longitude

:Missing Attribute Values: None

This dataset was obtained from the StatLib repository.
https://www.dcc.fc.up.pt/~ltorgo/Regression/

print(fe_cal.feature_names) #特征列的名字
MedInc - 中位收入（Median Income）
HouseAge - 房屋年龄（House Age）
AveRooms - 平均房间数（Average Number of Rooms）
AveBedrms - 平均卧室数（Average Number of Bedrooms）
Population - 人口数量（Population）
AveOccup - 平均居住人数（Average Occupancy）
Latitude - 纬度（Latitude）
Longitude - 经度（Longitude）

In [4]:
fe_cal.target.shape  # 目标数据的数量

(20640,)

In [18]:
# 分割训练集和测试集
x_train,x_test,y_train,y_test =train_test_split(fe_cal.data,fe_cal.target,test_size=0.25,random_state=1)

# 查看训练集的大小
print(x_train.shape)

# 对特征值进行标准化
std_x = StandardScaler()
x_train = std_x.fit_transform(x_train)  # 训练集标准化
x_test = std_x.transform(x_test)  # 测试集标准化
print(y_train.shape)

# # 目标值进行了标准化，暂时没有对目标值进行标准化处理
# std_y = StandardScaler()
# #
# # #标签进行标准化
# # 目标值是一维的，这里需要传进去2维的
# y_train = std_y.fit_transform(y_train.reshape(-1, 1))
# print(y_train.shape)

(15480, 8)
(15480,)


# 未对目标值进行标准化处理的训练代码

In [19]:
import os
# 正规方程求解方式预测结果，正规方程进行线性回归
lr = LinearRegression()
# 对模型进行训练
lr.fit(x_train,y_train)
print('回归系数',lr.coef_)

y_predict =lr.predict(x_test)
# 保存训练好的模型，模型中保存的是w的值，也保存了模型结构
# 保存模型放在fit之后即可
os.unlink('./tmp/test.pkl') # 删除之前的模型文件
joblib.dump(lr, "./tmp/test.pkl")
print('正规方程测试集里面每个房子的预测价格',y_predict[:10])
print('正规方程的均方误差',mean_squared_error(y_test,y_predict))

回归系数 [ 0.83167028  0.12159502 -0.26758589  0.30983997 -0.00518054 -0.04040421
 -0.90736902 -0.88212727]
正规方程测试集里面每个房子的预测价格 [2.12391852 0.93825754 2.7088455  1.70873764 2.82954754 3.50376456
 3.0147162  1.62781292 1.74317518 2.01897806]
正规方程的均方误差 0.5356532845422556


# 对目标值进行标准化处理的训练代码

In [15]:
lr = LinearRegression()
lr.fit(x_train,y_train)
print('回归系数', lr.coef_)
y_predict = lr.predict(x_test)
y_lr_predict = std_y.inverse_transform(y_predict)  # 进行反标准化，将标准化后的值转换成原来的值,因为y_test时没有进行标准化的，所以要将y_train进行反标准化
#保存模型放在fit之后即可
os.unlink('./tmp/test.pkl') # 删除之前的模型文件
joblib.dump(lr, "./tmp/test.pkl")
print("正规方程测试集里面每个房子的预测价格：", y_lr_predict[0:10])
print("正规方程的均方误差：", mean_squared_error(y_test, y_lr_predict))

回归系数 [[ 0.71942632  0.10518431 -0.23147194  0.26802332 -0.00448136 -0.03495117
  -0.7849086  -0.76307353]]
正规方程测试集里面每个房子的预测价格： [[2.12391852]
 [0.93825754]
 [2.7088455 ]
 [1.70873764]
 [2.82954754]
 [3.50376456]
 [3.0147162 ]
 [1.62781292]
 [1.74317518]
 [2.01897806]]
正规方程的均方误差： 0.5356532845422556


In [16]:
#模拟上线时加载模型
model = joblib.load("./tmp/test.pkl")
# # 因为目标值进行了标准化，一定要把预测后的值逆向转换回来
y_predict = model.predict(x_test)

#
# print("保存的模型预测的结果：", y_predict[0:10])
# print("正规方程的均方误差：", mean_squared_error(y_test, y_predict))
print("保存的y标准化后的模型预测的结果：", std_y.inverse_transform(y_predict)[0:10])
print("正规方程inverse后的均方误差：", mean_squared_error(y_test,
                                               std_y.inverse_transform(y_predict)))

保存的y标准化后的模型预测的结果： [[2.12391852]
 [0.93825754]
 [2.7088455 ]
 [1.70873764]
 [2.82954754]
 [3.50376456]
 [3.0147162 ]
 [1.62781292]
 [1.74317518]
 [2.01897806]]
正规方程inverse后的均方误差： 0.5356532845422556


# 3 线性回归之梯度下降进行房价预测

In [20]:
# 梯度下降去进行房价预测,数据量大要用这个
# learning_rate的不同方式，代表学习率变化的算法不一样,比如constant,invscaling,adaptive
# 默认可以去调 eta0 = 0.008，会改变learning_rate的初始值
# learning_rate='optimal',alpha是正则化力度，但是会影响学习率的值，由alpha来算学习率
# penalty代表正则化，分为l1和l2
# eta0=0.01, penalty='l2',max_iter=1000最大迭代次数
sgd = SGDRegressor(eta0=0.01,penalty='l2', max_iter=1000)
# # 训练
sgd.fit(x_train, y_train)
#
print('梯度下降的回归系数', sgd.coef_)
#
# 预测测试集的房子价格
# y_sgd_predict = std_y.inverse_transform(sgd.predict(x_test).reshape(-1, 1))
y_predict = sgd.predict(x_test)
# print("梯度下降测试集里面每个房子的预测价格：", y_sgd_predict)
print("梯度下降的均方误差：", mean_squared_error(y_test, y_predict))
# print("梯度下降的原始房价量纲均方误差：", mean_squared_error(std_y.inverse_transform(y_test), y_sgd_predict))

梯度下降的回归系数 [ 0.81159549  0.14662228 -0.21861573  0.35197226 -0.00484849 -0.0397017
 -0.88318335 -0.88800914]
梯度下降的均方误差： 0.5412611480513019


In [21]:
w=1
learning_rate=0.1  #这里是学习率，可以调节
def loss(w):
    return 3*w**2+2*w+2
def dao_shu(w):
    return 6*w+2
for i in range(30):
    w=w-learning_rate*dao_shu(w)
    print(f'w {w} 损失{loss(w)}')

w 0.19999999999999996 损失2.5199999999999996
w -0.12000000000000005 损失1.8032
w -0.24800000000000003 损失1.688512
w -0.2992 损失1.67016192
w -0.31968 损失1.6672259072
w -0.327872 损失1.666756145152
w -0.33114879999999997 损失1.6666809832243201
w -0.33245952 损失1.6666689573158913
w -0.332983808 损失1.6666670331705427
w -0.3331935232 损失1.6666667253072869
w -0.33327740928 损失1.666666676049166
w -0.333310963712 损失1.6666666681678666
w -0.3333243854848 损失1.6666666669068586
w -0.33332975419392 损失1.6666666667050973
w -0.333331901677568 损失1.6666666666728156
w -0.3333327606710272 损失1.6666666666676506
w -0.3333331042684109 损失1.6666666666668242
w -0.33333324170736434 损失1.6666666666666918
w -0.33333329668294576 损失1.6666666666666707
w -0.3333333186731783 损失1.6666666666666674
w -0.3333333274692713 损失1.6666666666666667
w -0.3333333309877085 损失1.6666666666666667
w -0.3333333323950834 损失1.6666666666666665
w -0.33333333295803336 损失1.6666666666666667
w -0.33333333318321334 损失1.6666666666666667
w -0.33333333327328535 损失1.6

# 4 岭回归

In [23]:
# 岭回归进行房价预测
# 岭回归是对线性回归加入L2正则化，L2正则化是对系数平方和进行惩罚
rd =Ridge(alpha = 0.2)  # alpha是正则化强度
rd.fit(x_train,y_train)
print(rd.coef_)
# 预测测试集的房子的价格
print(rd.predict(x_test).shape)
y_predict = rd.predict(x_test)
print("岭回归的均方误差：", mean_squared_error(y_test, y_predict))

[ 0.83166376  0.12161284 -0.26755063  0.30979361 -0.00517436 -0.04040534
 -0.90720032 -0.88195708]
(5160,)
岭回归的均方误差： 0.5356516224099191


# 5 lasso回归

In [25]:
# Lasso回归去进行房价预测
#alpha就是补偿的系数
print(x_train.shape)
print(y_train.shape)
ls = Lasso(alpha=0.001)

ls.fit(x_train, y_train)

print(ls.coef_)
#
# # 预测测试集的房子价格
print(ls.predict(x_test).shape)
print('-'*50)
# y_ls_predict = std_y.inverse_transform(ls.predict(x_test).reshape(-1,1))
y_predict = ls.predict(x_test)
# print("Lasso回归里面每个房子的预测价格：", y_rd_predict)
#
print("Lasso回归的均方误差：", mean_squared_error(y_test, y_predict))
# print("Lasso回归的均方误差：", mean_squared_error(std_y.inverse_transform(y_test), y_ls_predict))

(15480, 8)
(15480,)
[ 0.82655827  0.1225482  -0.25369194  0.29596304 -0.00381001 -0.03948424
 -0.89646842 -0.87060253]
(5160,)
--------------------------------------------------
Lasso回归的均方误差： 0.5356324125105497


# 逻辑回归

In [46]:
"""
逻辑回归做二分类进行癌症预测（根据细胞的属性特征）
"""

# 构造列标签名
column =['Sample code number', 'Clump Thickness', 'Uniformity of Cell Size', 'Uniformity of Cell Shape','Marginal Adhesion', 'Single Epithelial Cell Size', 'Bare Nuclei', 'Bland Chromatin', 'Normal Nucleoli','Mitoses', 'Class']

data =pd.read_csv('.//data//breast-cancer-wisconsin.csv',names = column)
print(data.info())
print('-'*50)
data.describe(include='all')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 699 entries, 0 to 698
Data columns (total 11 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   Sample code number           699 non-null    int64 
 1   Clump Thickness              699 non-null    int64 
 2   Uniformity of Cell Size      699 non-null    int64 
 3   Uniformity of Cell Shape     699 non-null    int64 
 4   Marginal Adhesion            699 non-null    int64 
 5   Single Epithelial Cell Size  699 non-null    int64 
 6   Bare Nuclei                  699 non-null    object
 7   Bland Chromatin              699 non-null    int64 
 8   Normal Nucleoli              699 non-null    int64 
 9   Mitoses                      699 non-null    int64 
 10  Class                        699 non-null    int64 
dtypes: int64(10), object(1)
memory usage: 60.2+ KB
None
--------------------------------------------------


,Sample code number,Clump Thickness,Uniformity of Cell Size,Uniformity of Cell Shape,Marginal Adhesion,Single Epithelial Cell Size,Bare Nuclei,Bland Chromatin,Normal Nucleoli,Mitoses,Class
count,6.990000e+02,699.000000,699.000000,699.000000,699.000000,699.000000,699,699.000000,699.000000,699.000000,699.000000
unique,NaN,NaN,NaN,NaN,NaN,NaN,11,NaN,NaN,NaN,NaN
top,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN
freq,NaN,NaN,NaN,NaN,NaN,NaN,402,NaN,NaN,NaN,NaN
mean,1.071704e+06,4.417740,3.134478,3.207439,2.806867,3.216023,NaN,3.437768,2.866953,1.589413,2.689557
std,6.170957e+05,2.815741,3.051459,2.971913,2.855379,2.214300,NaN,2.438364,3.053634,1.715078,0.951273
min,6.163400e+04,1.000000,1.000000,1.000000,1.000000,1.000000,NaN,1.000000,1.000000,1.000000,2.000000
25%,8.706885e+05,2.000000,1.000000,1.000000,1.000000,2.000000,NaN,2.000000,1.000000,1.000000,2.000000
50%,1.171710e+06,4.000000,1.000000,1.000000,1.000000,2.000000,NaN,3.000000,1.000000,1.000000,2.000000
75%,1.238298e+06,6.000000,5.000000,5.000000,4.000000,4.000000,NaN,5.000000,4.000000,1.000000,4.000000


In [47]:
# 由于发现Bare Nuclei为object类型，所以我们查看他的数据
print(data['Bare Nuclei'].unique())

['1' '10' '2' '4' '3' '9' '7' '?' '5' '8' '6']


In [48]:
# 将Bare Nuclei的？替换成nan
data = data.replace(to_replace='?',value = np.nan)
# 删除有nan的那一行
data = data.dropna()
print(data.shape)

(683, 11)


In [49]:
column[10]

'Class'

In [50]:
data[column[10]].unique()

array([2, 4], dtype=int64)

In [51]:
#把第6列的字符串转化为数字类型
data[column[6]] = data[column[6]].astype('int16')

In [52]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 683 entries, 0 to 698
Data columns (total 11 columns):
 #   Column                       Non-Null Count  Dtype
---  ------                       --------------  -----
 0   Sample code number           683 non-null    int64
 1   Clump Thickness              683 non-null    int64
 2   Uniformity of Cell Size      683 non-null    int64
 3   Uniformity of Cell Shape     683 non-null    int64
 4   Marginal Adhesion            683 non-null    int64
 5   Single Epithelial Cell Size  683 non-null    int64
 6   Bare Nuclei                  683 non-null    int16
 7   Bland Chromatin              683 non-null    int64
 8   Normal Nucleoli              683 non-null    int64
 9   Mitoses                      683 non-null    int64
 10  Class                        683 non-null    int64
dtypes: int16(1), int64(10)
memory usage: 60.0 KB


In [56]:
# 进行数据分割
x_train,x_test,y_train,y_test = train_test_split(data[column[1:10]],data[column[10]],test_size=0.25,random_state=1)

# 进行表转化处理
std =StandardScaler()

# 进行标准化
x_train = std.fit_transform(x_train)
x_test = std.transform(x_test)

In [58]:
# 逻辑回归预测
# C正则化力度,跟学习率有关
# solver = 'liblinear'  solver是学习率优化算法，就是学习率会随着epoch的变化而变化
#epoch就代表第几次迭代
#max_iter 最大迭代次数
lg =LogisticRegression(C = 0.5,solver = 'lbfgs')
lg.fit(x_train,y_train)
print(lg.coef_)

y_predict = lg.predict(x_test)
# print(y_predict) #预测的标签
print("准确率：", lg.score(x_test, y_test))
print(y_test[0:5])
print('-'*50)
print(lg.predict_proba(x_test)[0:5])  #得出对应分类的概率

[[1.11400191 0.25293086 0.78938469 0.60986034 0.0728013  1.10834397
  0.7794668  0.64312128 0.67692658]]
准确率： 0.9824561403508771
444    2
24     2
195    2
49     4
375    2
Name: Class, dtype: int64
--------------------------------------------------
[[0.94893919 0.05106081]
 [0.99494175 0.00505825]
 [0.98365149 0.01634851]
 [0.02707911 0.97292089]
 [0.99732446 0.00267554]]


In [59]:
# macro avg 平均值  weighted avg 加权平均值
print(classification_report(y_test, y_predict, labels=[2, 4], target_names=["良性", "恶性"]))
#AUC计算要求是二分类，不需要是0和1
print("AUC指标：", roc_auc_score(y_test, y_predict))

              precision    recall  f1-score   support

          良性       0.97      1.00      0.99       111
          恶性       1.00      0.95      0.97        60

    accuracy                           0.98       171
   macro avg       0.99      0.97      0.98       171
weighted avg       0.98      0.98      0.98       171

AUC指标： 0.975
